# 🔁 LUMI PIPELINE — parametros_lumi_dbm v10 (con MetaCycle / meta2d)

> Procesa automáticamente todas las carpetas con `lumi` en el nombre,
> extrae **TODOS** los parámetros (HP, LP, RAW_T, WV, derivadas, cos picos, FR)
> y genera un único DataFrame consolidado `dbm`.

### ⚠️ Cambio importante respecto al notebook anterior (dbm_v7):
El bucle anterior usaba `dbm.join(otro_df)` indexado solo por **(condition, variable)**.
Como una misma combinación (cond, var) aparece en VARIOS experimentos
(p.ej. `(Blank, A1)` está en los 7 experimentos), el join hacía un
**producto cartesiano** y multiplicaba las filas — y además mezclaba valores
entre experimentos distintos.

**Solución**: el índice ahora es triple → `(experiment, condition, variable)`.
Así cada fila es **única** y `join` se comporta como un merge sano.

In [ ]:
# ─── IMPORTS ──────────────────────────────────────────────────────────────────
import os
import sys
import pandas as pd

# ─── PATH: directorio de trabajo donde están las carpetas lumi + ma_methods ──
BASE_DIR = r"C:/Users/neren/OneDrive/Documentos/TFG_MUNICH/Python_notebooks"
sys.path.insert(0, BASE_DIR)

from ma_methods10 import DataProcessing, parametros_ondas

In [ ]:
# ─── FUNCIÓN: importar datos según extensión ──────────────────────────────────
def importer(file_path):
    if file_path.endswith(('.txt', '.tsv')):
        return pd.read_csv(file_path, sep='\t')
    elif file_path.endswith('.csv'):
        return pd.read_csv(file_path)
    elif file_path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(file_path)

## 🆕 Helper: `load_one_folder`

> Esto es **nuevo respecto a tu v7**. Antes, cada bloque (wave, cos, derivative, etc.)
> repetía las MISMAS 30 líneas de carga (importer + layout + melt + treatment).
> Lo encapsulo en una función para que las celdas de cada bloque sean más cortas
> y claras. **Si necesitas tocar la carga, ahora hay UN solo sitio donde editar**.

In [ ]:
# ─── HELPER: carga + melt + treatment de UNA carpeta ──────────────────────────
def load_one_folder(folder_name):
    """
    Procesa una carpeta lumi y devuelve:
      - data_melt        : DataFrame MELT con datos RAW [time, variable, value, condition]
      - data_treat_melt  : DataFrame MELT con datos tratados (detrend + zscore)
      - exp_id           : int con los 4 primeros dígitos del nombre de carpeta
    """
    folder_path = os.path.join(BASE_DIR, folder_name)

    # ── 1. Identificar archivos ──────────────────────────────────────────────
    files = [f for f in os.listdir(folder_path)
             if not os.path.isdir(os.path.join(folder_path, f))
             and not f.endswith('.ipynb')]
    layout_file = os.path.join(folder_path,
        [f for f in files if 'layout' in f.lower()][0])
    data_file = os.path.join(folder_path,
        [f for f in files
         if 'lumi' in f.lower() and 'layout' not in f.lower()
         and f.lower().endswith(('.csv', '.xlsx', '.xls', '.txt', '.tsv'))][0])

    # ── 2. Cargar datos ──────────────────────────────────────────────────────
    data = importer(data_file)
    data.columns = data.columns.str.strip()

    # ── 3. Layout → id_dict ──────────────────────────────────────────────────
    layout_df   = pd.read_excel(layout_file)
    layout_melt = layout_df.melt(id_vars=layout_df.columns[0]).astype(str)
    layout_melt['id'] = (layout_melt[layout_melt.columns[0]]
                         + layout_melt[layout_melt.columns[1]])
    id_dict = dict(zip(layout_melt['id'], layout_melt['value']))

    # ── 4. Data MELT (RAW) ───────────────────────────────────────────────────
    data_melt = data.melt(id_vars='time')
    data_melt.columns = ['time', 'variable', 'value']
    data_melt['condition'] = data_melt['variable'].replace(id_dict)
    data_melt[['time', 'value']] = data_melt[['time', 'value']].astype(float)

    # ── 5. Tratamiento + MELT ────────────────────────────────────────────────
    data_treat = DataProcessing.pipeline(data.iloc[:, 1:])
    data_treat.index.name = 'time'
    data_treat_melt = data_treat.reset_index().melt(id_vars='time')
    data_treat_melt['condition'] = data_melt['variable'].replace(id_dict)

    # ── 6. Identificador de experimento (4 primeros dígitos como int) ────────
    exp_id = int(folder_name[:4])

    return data_melt, data_treat_melt, exp_id

## Bloque 0 — Bucle principal (HP, LP, RAW_T)

> Idéntico en lógica al notebook anterior, **excepto en una línea clave**:
> añado `experiment` al `set_index` ANTES de hacer el join.
> Eso da un índice triple `(experiment, condition, variable)` que es **único**.

In [ ]:
# ─── BUCLE PRINCIPAL ──────────────────────────────────────────────────────────
db = pd.DataFrame()

# Filtrar solo carpetas (no archivos) con "lumi" en el nombre
lumi_folders = sorted([
    f for f in os.listdir(BASE_DIR)
    if os.path.isdir(os.path.join(BASE_DIR, f)) and "lumi" in f.lower()
])

print(f"📂 Carpetas lumi encontradas: {lumi_folders}\n")

for folder_name in lumi_folders:
    print(f"▶ Procesando: {folder_name}")
    try:
        # ── Carga + melt + treatment (usa el helper) ─────────────────────────
        data_melt, data_treat_melt, exp_id = load_one_folder(folder_name)

        # ── División por t=125h (entrainment / free run) ─────────────────────
        data_ent       = data_melt[data_melt["time"] < 125]
        data_treat_ent = data_treat_melt[data_treat_melt["time"] < 125]

        # ── Parámetros HP, LP, RAW_T ─────────────────────────────────────────
        high_p = parametros_ondas.high_low_peaks_stats(data_treat_ent, kind="high")
        low_p  = parametros_ondas.high_low_peaks_stats(data_treat_ent, kind="low")
        trend  = parametros_ondas.trend_stats(data_ent)

        # ── Prefijos ─────────────────────────────────────────────────────────
        high_p.columns = [c if c in ("condition", "variable") else f"HP_{c}"    for c in high_p.columns]
        low_p.columns  = [c if c in ("condition", "variable") else f"LP_{c}"    for c in low_p.columns]
        trend.columns  = [c if c in ("condition", "variable") else f"RAW_T_{c}" for c in trend.columns]

        # ⭐ CLAVE: añadir experiment como columna ANTES de set_index
        for d in (high_p, low_p, trend):
            d["experiment"] = exp_id

        # ⭐ CLAVE: índice TRIPLE (experiment, condition, variable) → ÚNICO
        high_p = high_p.set_index(["experiment", "condition", "variable"])
        low_p  = low_p.set_index(["experiment", "condition", "variable"])
        trend  = trend.set_index(["experiment", "condition", "variable"])

        # ── Join entre los tres parámetros del MISMO experimento ─────────────
        temp = high_p.join(low_p).join(trend)
        db = pd.concat([db, temp])
        print(f"   ✅ OK — {len(temp)} filas añadidas")

    except Exception as e:
        print(f"   ❌ ERROR en '{folder_name}': {e}")
        continue

print(f"\n✅ Pipeline completado. db shape: {db.shape}")
print(f"   Index único: {db.index.is_unique}")  # debe ser True

In [ ]:
# ─── PREVIEW del db ───────────────────────────────────────────────────────────
db

### Averiguar procedencia de los valores NaN

In [ ]:
# ─── 1. ¿LOS NaN SIEMPRE APARECEN JUNTOS? ────────────────────────────────────
nan_cols = db.columns[db.isnull().any(axis=0)]

print("Correlación de NaN entre columnas afectadas:")
print("(True = esa fila tiene NaN en ambas columnas)\n")
print(db[nan_cols].isnull().corr().to_string())

# ─── 2. RECUENTO POR condition Y experiment ───────────────────────────────────
nan_mask = db.isnull().any(axis=1)

recuento = (
    db[nan_mask]
    .reset_index()
    [["condition", "experiment"]]
    .value_counts()
    .reset_index()
    .rename(columns={0: "n_NaN"})
)
recuento

### Cambiar valores str por numéricos (`dbn`)

In [ ]:
# ─── ENCODING DE COLUMNAS CATEGÓRICAS ────────────────────────────────────────
# dbn ==> n de "cambio de int a números"

trend_map = {
    "decrease" : -1,
    "no trend" :  0,
    "increase" :  1
}

strength_map = {
    "no trend" :  0,
    "weak"     :  1,
    "moderate" :  2,
    "strong"   :  3
}

dbn = db.copy()
dbn["RAW_T_trend"]          = dbn["RAW_T_trend"].map(trend_map)
dbn["RAW_T_trend_strength"] = dbn["RAW_T_trend_strength"].map(strength_map)

# ─── VERIFICACIÓN ─────────────────────────────────────────────────────────────
print("Valores únicos RAW_T_trend:         ", sorted(dbn["RAW_T_trend"].dropna().unique()))
print("Valores únicos RAW_T_trend_strength:", sorted(dbn["RAW_T_trend_strength"].dropna().unique()))

dbn[["RAW_T_trend", "RAW_T_trend_strength"]]

### Extracción de metadatos: reporter, gene, medium, entrainment (`dbm`)

In [ ]:
# ─── EXTRACCIÓN DE METADATOS: reporter, gene, medium, entrainment ────────────
# dbm ==> m de "más columnas"

import re

# ── Listas de referencia ──────────────────────────────────────────────────────
GENES = [
    "ytva_WT",  "eps_WT",
    "ytva_X", "eps_X",
    "ytva_Y", "eps_Y",
    "ytva_frn",  "eps_frn",
    "ytva_lct",  "eps_lct"
]

MEDIUMS = [
    "NSMP_NO3_Glu", "SM_Glu_NO3",
    "NSMP_NO3",     "NSMP_Glu",
    "SM_Glu",       "SM_NO3",
    "NSMP",         "SM"
]

# ── Funciones ─────────────────────────────────────────────────────────────────
def get_reporter(condition):
    c = str(condition)
    if re.search(r'ytva', c, re.IGNORECASE):   return "ytva"
    elif re.search(r'eps', c, re.IGNORECASE):  return "eps"
    else:                                       return "Blank"

def get_gene(condition):
    c = str(condition).replace(" ", "")
    if re.search(r'ytva', c, re.IGNORECASE):   reporter = "ytva"
    elif re.search(r'eps', c, re.IGNORECASE):  reporter = "eps"
    else:                                       return "Blank"
    if re.search(r'X', c, re.IGNORECASE):   return f"{reporter}_X"
    elif re.search(r'Y', c, re.IGNORECASE): return f"{reporter}_Y"
    elif re.search(r'frn',  c, re.IGNORECASE): return f"{reporter}_frn"
    elif re.search(r'lct',  c, re.IGNORECASE): return f"{reporter}_lct"
    else:                                       return f"{reporter}_WT"

def get_medium(condition):
    c = str(condition).replace(" ", "")
    for medium in MEDIUMS:
        if re.search(re.escape(medium), c, re.IGNORECASE):
            return medium
    return None

def get_entrainment(folder_name):
    if re.search(r'LightDD', folder_name):  return "LightDD"
    elif re.search(r'Light', folder_name):  return "Light"
    elif re.search(r'Temp', folder_name):   return "Temperature"
    return None

# ── Aplicar ───────────────────────────────────────────────────────────────────
conditions = dbn.index.get_level_values("condition")
dbn["reporter"]    = conditions.map(get_reporter)
dbn["genome"]      = conditions.map(get_gene)
dbn["medium"]      = conditions.map(get_medium)
# ⭐ CAMBIO: "experiment" está en el ÍNDICE, no en columnas → usamos get_level_values
dbn["entrainment"] = dbn.index.get_level_values("experiment").map(
    {int(f[:4]): get_entrainment(f) for f in lumi_folders}
)

# ── Reordenar columnas ────────────────────────────────────────────────────────
meta_cols  = ["reporter", "genome", "medium", "entrainment"]
rest_cols  = [c for c in dbn.columns if c not in meta_cols]
dbm = dbn[meta_cols + rest_cols]
dbm

## Cambiar nombres `X` → `x`, `Y` → `y`

In [ ]:
# ─── RENOMBRADO: X → x  |  Y → y ────────────────────────────────────────
# Como ahora el index es TRIPLE (experiment, condition, variable),
# hay que reconstruirlo manteniendo los 3 niveles.


new_condition = (
    dbm.index.get_level_values("condition")
    .str.replace("X", "x", regex=False)
    .str.replace("Y", "y", regex=False)
)

dbm.index = pd.MultiIndex.from_arrays(
    [dbm.index.get_level_values("experiment"),
     new_condition,
     dbm.index.get_level_values("variable")],
    names=["experiment", "condition", "variable"]
)

# ── Columna: genome ───────────────────────────────────────────────────────────
dbm["genome"] = (
    dbm["genome"]
    .str.replace("X", "x", regex=False)
    .str.replace("Y", "y", regex=False)
)

print("✅ Renombrado completado")
print("  condition — ejemplo:", dbm.index.get_level_values("condition")[0])
print("  genome    — únicos: ", sorted(dbm["genome"].unique()))
dbm

## Helper para los siguientes bloques

> Los siguientes bloques añaden nuevas columnas haciendo un bucle por carpetas.
> Como el `dbm` actual tiene `X→x` y `Y→y`, hay que aplicar el mismo
> renombrado a las conditions del nuevo df ANTES del join,
> sino las keys no coinciden.

In [ ]:
# ─── helper: aplica renombrado X→x, Y→y a la columna condition ─────────
def _renombrar_condition(s):
    """Aplica el mismo renombrado que la celda de renombrado del dbm."""
    return s.str.replace('X', 'x', regex=False).str.replace('Y', 'y', regex=False)

## Bloque 1 — Parámetros wavelet (`WV_phase`, `WV_amplitude`, `WV_power`)

> Aquí es donde se rompía el v7. La corrección es **añadir `experiment` al set_index**
> antes de hacer el join. Como ahora `dbm` también tiene `experiment` en su index,
> el join es uno-a-uno.

In [ ]:
# ─── EXTRACCIÓN DE PARÁMETROS WAVELET: fase, amplitud, power ──────────────────
wave_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_ent = data_treat_melt[data_treat_melt['time'] < 125].copy()

        # ── Mismo renombrado X→x / Y→y que el dbm ──────────────────────
        data_treat_ent['condition'] = _renombrar_condition(data_treat_ent['condition'])

        wp = parametros_ondas.wave_params(data_treat_ent)
        if wp.empty:
            print(f'  ⚠️  wave_params vacío — {folder_name}')
            continue

        wp['experiment'] = exp_id   # ⭐ añadir experiment ANTES de set_index
        wave_rows.append(wp)
        print(f'  ✅ wave_params OK — {folder_name}  ({len(wp)} filas)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR wave_params {folder_name}: {e}')
        traceback.print_exc()

if not wave_rows:
    print('\n❌ wave_rows vacío — revisa los errores de arriba')
else:
    # ⭐ set_index TRIPLE → index único → join uno-a-uno
    wave_df = pd.concat(wave_rows).set_index(['experiment', 'condition', 'variable'])
    wave_cols = list(wave_df.columns)
    dbm = dbm.drop(columns=[c for c in wave_cols if c in dbm.columns]).join(wave_df)
    print(f'\n✅ Columnas wavelet añadidas: {wave_cols}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')
    print(f'   NaN en WV_phase: {dbm["WV_phase"].isna().sum()} / {len(dbm)}')

## Bloque 2 — Coseno de la fase + métricas (Pearson, Spearman, Cross-corr, DTW)

> Misma corrección: `experiment` en el index triple antes del join.

In [ ]:
# ─── COSENO DE LA FASE + MÉTRICAS: Pearson, Spearman, Cross-corr, DTW ────────
cos_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_ent = data_treat_melt[data_treat_melt['time'] < 125].copy()
        data_treat_ent['condition'] = _renombrar_condition(data_treat_ent['condition'])

        cp = parametros_ondas.phase_cosine_similarity(data_treat_ent)
        if cp.empty:
            print(f'  ⚠️  cos_similarity vacío — {folder_name}')
            continue

        cp['experiment'] = exp_id
        cos_rows.append(cp)
        print(f'  ✅ cos_similarity OK — {folder_name}  ({len(cp)} filas)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR cos_similarity {folder_name}: {e}')
        traceback.print_exc()

if not cos_rows:
    print('\n❌ cos_rows vacío — revisa los errores de arriba')
else:
    cos_df = pd.concat(cos_rows).set_index(['experiment', 'condition', 'variable'])
    cos_cols = list(cos_df.columns)
    dbm = dbm.drop(columns=[c for c in cos_cols if c in dbm.columns]).join(cos_df)
    print(f'\n✅ Columnas coseno/Pearson añadidas: {cos_cols}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')

## Bloque 3 — Derivadas de la señal (1ª, 2ª, 3ª) + asimetría

In [ ]:
# ─── EXTRACCIÓN DE DERIVADAS (1ª, 2ª, 3ª) + ASIMETRÍA SUBIDA/BAJADA ──────────
deriv_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_ent = data_treat_melt[data_treat_melt['time'] < 125].copy()
        data_treat_ent['condition'] = _renombrar_condition(data_treat_ent['condition'])

        deriv = parametros_ondas.derivative_params(data_treat_ent)
        if deriv.empty:
            print(f'  ⚠️  derivative_params vacío — {folder_name}')
            continue

        deriv['experiment'] = exp_id
        deriv_rows.append(deriv)
        print(f'  ✅ derivative_params OK — {folder_name}  ({len(deriv)} filas)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR derivative_params {folder_name}: {e}')
        traceback.print_exc()

if not deriv_rows:
    print('\n❌ deriv_rows vacío — revisa los errores de arriba')
else:
    deriv_df = pd.concat(deriv_rows).set_index(['experiment', 'condition', 'variable'])
    deriv_cols = list(deriv_df.columns)
    dbm = dbm.drop(columns=[c for c in deriv_cols if c in dbm.columns]).join(deriv_df)
    print(f'\n✅ Columnas derivadas añadidas: {deriv_cols}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')

## Bloque 4 — Hora circadiana de los picos del `cos(fase)`

In [ ]:
# ─── HORA CIRCADIANA DE LOS PICOS DEL cos(FASE) ──────────────────────────────
cos_peak_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_ent = data_treat_melt[data_treat_melt['time'] < 125].copy()
        data_treat_ent['condition'] = _renombrar_condition(data_treat_ent['condition'])

        cp = parametros_ondas.cos_phase_peak_time(data_treat_ent)
        if cp.empty:
            print(f'  ⚠️  cos_phase_peak_time vacío — {folder_name}')
            continue

        cp['experiment'] = exp_id
        cos_peak_rows.append(cp)
        print(f'  ✅ cos_phase_peak_time OK — {folder_name}  ({len(cp)} filas)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR cos_phase_peak_time {folder_name}: {e}')
        traceback.print_exc()

if not cos_peak_rows:
    print('\n❌ cos_peak_rows vacío — revisa los errores de arriba')
else:
    cos_pk_df = pd.concat(cos_peak_rows).set_index(['experiment', 'condition', 'variable'])
    cos_pk_cols = list(cos_pk_df.columns)
    dbm = dbm.drop(columns=[c for c in cos_pk_cols if c in dbm.columns]).join(cos_pk_df)
    print(f'\n✅ Columnas cos picos añadidas: {cos_pk_cols}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')

## Bloque 5 — Parámetros de free run (sobre `data_treat_fr`, t ≥ 125h)

In [ ]:
# ─── FREE RUN — PARÁMETROS DE RITMICIDAD v7 (prefijo FR_) ────────────────────
fr_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_fr = data_treat_melt[data_treat_melt['time'] >= 125].copy()
        data_treat_fr['condition'] = _renombrar_condition(data_treat_fr['condition'])

        fr = parametros_ondas.freerun_params(data_treat_fr)
        if fr.empty:
            print(f'  ⚠️  freerun_params vacío — {folder_name}')
            continue

        fr['experiment'] = exp_id
        fr_rows.append(fr)
        print(f'  ✅ freerun_params OK — {folder_name}  ({len(fr)} filas)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR freerun_params {folder_name}: {e}')
        traceback.print_exc()

if not fr_rows:
    print('\n❌ fr_rows vacío — revisa los errores arriba')
else:
    fr_df = pd.concat(fr_rows).set_index(['experiment', 'condition', 'variable'])
    fr_cols = list(fr_df.columns)
    dbm = dbm.drop(columns=[c for c in fr_cols if c in dbm.columns]).join(fr_df)
    print(f'\n✅ Columnas FR añadidas: {fr_cols}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')
    print(f'   Rítmicas (FR_rhythmic=True): '
          f'{dbm["FR_rhythmic"].sum()} / {dbm["FR_rhythmic"].notna().sum()}')

## 🆕 Bloque 6 — MetaCycle / meta2d (ritmicidad estadística)

> Llama al script **`run_meta2d.R`** (que debe estar en la misma carpeta que este notebook)
> a través de `subprocess`. Calcula 3 métricas estadísticas de ritmicidad combinando
> JTK_CYCLE, ARS y Lomb-Scargle, y añade 4 columnas nuevas al `dbm`:
>
> | Columna       | Significado |
> |---------------|-------------|
> | `MC_period`   | Periodo estimado (horas), restringido al rango 22-32h en `run_meta2d.R` |
> | `MC_pvalue`   | p-value integrado de los 3 métodos |
> | `MC_BH_Q`     | p-value corregido por multiple testing (Benjamini-Hochberg) |
> | `MC_rhythmic` | 1 si `MC_BH_Q < 0.001`, 0 si no |
>
> ### ⚙️ Requisitos en tu PC
> 1. R instalado (descarga desde https://cran.r-project.org/)
> 2. Paquete MetaCycle: abre R y ejecuta `install.packages("MetaCycle")`
> 3. El script `run_meta2d.R` debe estar en `BASE_DIR`
> 4. Define abajo la ruta de `Rscript.exe`

In [ ]:
# ─── CONFIGURACIÓN R ──────────────────────────────────────────────────────────
# ⚠️ AJUSTA esta ruta según tu instalación de R en Windows
#    Para encontrarla: abre la consola de R y ejecuta R.home("bin")
RSCRIPT_PATH = r"C:/Program Files/R/R-4.6.0/bin/Rscript.exe"   # ← editar si tu versión es distinta
R_SCRIPT     = os.path.join(BASE_DIR, "run_meta2d.R")

# ─── Verificación rápida de que R está accesible ──────────────────────────────
import subprocess
try:
    chk = subprocess.run([RSCRIPT_PATH, "--version"], capture_output=True, text=True)
    print("✅ R encontrado:", chk.stderr.strip().split(chr(10))[0])
except FileNotFoundError:
    print(f"❌ No encuentro Rscript en: {RSCRIPT_PATH}")
    print(f"   Edita RSCRIPT_PATH arriba con la ruta correcta")
    raise

In [ ]:
# ─── EXTRACCIÓN DE PARÁMETROS METACYCLE / meta2d ──────────────────────────────
mc_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_fr = data_treat_melt[data_treat_melt['time'] >= 125].copy()
        data_treat_fr['condition'] = _renombrar_condition(data_treat_fr['condition'])

        mc = parametros_ondas.metacycle_params(
            data_treat_fr,
            rscript_path=RSCRIPT_PATH,
            r_script=R_SCRIPT,
        )
        if mc.empty:
            print(f'  ⚠️  metacycle_params vacío — {folder_name}')
            continue

        mc['experiment'] = exp_id
        mc_rows.append(mc)
        print(f'  ✅ metacycle_params OK — {folder_name}  ({len(mc)} pocillos)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR metacycle_params {folder_name}: {e}')
        traceback.print_exc()

if not mc_rows:
    print('\n❌ mc_rows vacío — revisa los errores arriba')
else:
    mc_df = pd.concat(mc_rows).set_index(['experiment', 'condition', 'variable'])
    mc_cols = list(mc_df.columns)
    dbm = dbm.drop(columns=[c for c in mc_cols if c in dbm.columns]).join(mc_df)

    # ── Calcular columna booleana MC_rhythmic ─────────────────────────────────
    BH_Q_THRESHOLD = 0.001   # ← cambia aquí si quieres otro umbral
    dbm['MC_rhythmic'] = (dbm['MC_BH_Q'] < BH_Q_THRESHOLD).astype(int)

    print(f'\n✅ Columnas MetaCycle añadidas: {mc_cols + ["MC_rhythmic"]}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')
    print(f'   Pocillos rítmicos (MC_BH_Q < {BH_Q_THRESHOLD}): '
          f'{int(dbm["MC_rhythmic"].sum())} / {len(dbm)} '
          f'({100*dbm["MC_rhythmic"].mean():.1f}%)')

## 🆕 Bloque 7 — Picos en free run (dos criterios de prominencia)

> Llama a la nueva función `parametros_ondas.fr_peaks_stats()` definida en
> `ma_methods10.py`. Es la réplica de `HP_n_peaks`, `HP_mean_peak_value`,
> `HP_std_peak_value` pero calculada sobre la ventana de **free run** (t ≥ 125 h).
>
> Cada pocillo se analiza con **dos criterios de prominencia** para que puedas
> comparar y elegir el que mejor te encaje:
>
> | Sufijo | Prominencia | Origen |
> |---|---|---|
> | *(sin sufijo)* | `1.5 × mean(|signal|)` | mismo criterio que `HP_` (entrainment) |
> | `_02` | `0.20 × max(|signal|)` | nuevo criterio escalado al máximo |
>
> Columnas añadidas al `dbm` (todas al **principio** del bloque FR_):
> `FR_n_peaks`, `FR_mean_peak_value`, `FR_std_peak_value`,
> `FR_n_peaks_02`, `FR_mean_peak_value_02`, `FR_std_peak_value_02`.
>
> Los pocillos sin picos detectados reciben **0** en todas las columnas
> (no `NaN`).

In [ ]:
# ─── PICOS EN FREE RUN — 2 CRITERIOS DE PROMINENCIA ──────────────────────────
peaks_rows = []

for folder_name in lumi_folders:
    try:
        _, data_treat_melt, exp_id = load_one_folder(folder_name)
        data_treat_fr = data_treat_melt[data_treat_melt['time'] >= 125].copy()
        data_treat_fr['condition'] = _renombrar_condition(data_treat_fr['condition'])

        # ── Llamada a la nueva función de ma_methods10 ────────────────────────
        pk = parametros_ondas.fr_peaks_stats(data_treat_fr)
        if pk.empty:
            print(f'  ⚠️  fr_peaks_stats vacío — {folder_name}')
            continue

        pk['experiment'] = exp_id
        peaks_rows.append(pk)
        print(f'  ✅ fr_peaks_stats OK — {folder_name}  ({len(pk)} pocillos)')

    except Exception as e:
        import traceback
        print(f'  ❌ ERROR fr_peaks_stats {folder_name}: {e}')
        traceback.print_exc()

if not peaks_rows:
    print('\n❌ peaks_rows vacío — revisa los errores arriba')
else:
    peaks_df = (pd.concat(peaks_rows)
                .set_index(['experiment', 'condition', 'variable']))
    peaks_cols = list(peaks_df.columns)

    # Limpiar columnas duplicadas si las hubiera (re-ejecución segura)
    dbm = dbm.drop(columns=[c for c in peaks_cols if c in dbm.columns]).join(peaks_df)

    # ── Reordenar: las 6 nuevas al PRINCIPIO del bloque FR_ ──────────────────
    todas_fr  = [c for c in dbm.columns if c.startswith('FR_')]
    otras_fr  = [c for c in todas_fr if c not in peaks_cols]
    no_fr     = [c for c in dbm.columns if not c.startswith('FR_')]
    mc_cols   = [c for c in no_fr if c.startswith('MC_')]
    sin_fr_mc = [c for c in no_fr if not c.startswith('MC_')]
    dbm = dbm[sin_fr_mc + peaks_cols + otras_fr + mc_cols]

    print(f'\n✅ Columnas añadidas: {peaks_cols}')
    print(f'   dbm shape: {dbm.shape}  |  index único: {dbm.index.is_unique}')
    print(f'\n📊 Estadísticas de las 6 columnas nuevas:')
    print(dbm[peaks_cols].describe().round(3))

## ✅ Verificación final

> Si todo va bien deberías ver:
> - **672 filas exactas** (96 pocillos × 7 experimentos)
> - **Index único = True** (sin duplicados)
> - Una columna por cada grupo: HP_, LP_, RAW_T_, WV_, D1_, D2_, D3_, COS_, FR_

In [ ]:
# ─── VERIFICACIÓN FINAL ──────────────────────────────────────────────────────
print(f"Shape final:           {dbm.shape}")
print(f"Index único:           {dbm.index.is_unique}")
print(f"Total filas:           {len(dbm)}")
print(f"Experiments:           {sorted(dbm.index.get_level_values('experiment').unique())}")
print(f"# experiments:         {dbm.index.get_level_values('experiment').nunique()}")
print()
print("Columnas por grupo:")
for prefix in ['HP_', 'LP_', 'RAW_T_', 'WV_', 'D1_', 'D2_', 'D3_', 'COS_', 'FR_', 'MC_']:
    n = sum(1 for c in dbm.columns if c.startswith(prefix))
    print(f"  {prefix:10} → {n} columnas")

## 🆕 Bloque 8 — Clasificación con etiquetas de texto

> Llama a `parametros_ondas.classify_fr_params()` definida en `ma_methods10.py`.
> Añade **4 columnas con etiquetas de texto** que clasifican parámetros de
> ritmicidad para facilitar análisis cualitativos y visualizaciones.
>
> ### Esquemas aplicados (todos umbrales fijos)
>
> **`FRc_wv_period_w`** ← `FR_wv_period_w` (período wavelet, h):
>
> | Rango | Etiqueta |
> |---|---|
> | [16, 20) | "muy corto" |
> | [20, 24) | "corto" |
> | [24, 28) | "largo" |
> | [28, 32] | "muy largo" |
>
> **`FRc_n_peaks`** ← `FR_n_peaks`:
>
> | Picos | Etiqueta |
> |---|---|
> | 0-1 | "ninguno/muy pocos" |
> | 2-3 | "pocos" |
> | 4-6 | "bien" |
> | 7-10 | "muchos" |
> | ≥ 11 | "ruido" |
>
> **`FRc_autocorr_val`** ← `FR_autocorr_val` (**percentiles** Q1, Q2, Q3 de tu distribución):
>
> | Rango | Etiqueta |
> |---|---|
> | < Q1 | "bajo" (25% inferior) |
> | [Q1, Q2) | "medio-bajo" |
> | [Q2, Q3) | "medio-alto" |
> | ≥ Q3 | "alto" (25% superior) |
>
> *Etiquetas relativas a tu propia distribución — los cuartiles se imprimen al ejecutar para que puedas citarlos en el TFG.*
>
> **`MCc_rhythmic`** ← `MC_rhythmic` (boolean):
>
> | Valor | Etiqueta |
> |---|---|
> | 0 | "Falso" |
> | 1 | "Verdadero" |
>
> *Notas:*
> - Las columnas numéricas originales se preservan. Las nuevas son `string` (categóricas de texto).
> - Para ML, sigue usando las numéricas originales (`FR_n_peaks`, `FR_wv_period_w`, etc.). Las nuevas `FRc_*` y `MCc_*` son para análisis cualitativos, plots y tablas resumen.

In [ ]:
# ─── CLASIFICACIÓN CON ETIQUETAS DE TEXTO ────────────────────────────────────
dbm = parametros_ondas.classify_fr_params(
    dbm,
    period_col='FR_wv_period_w',
    n_peaks_col='FR_n_peaks',
    autocorr_col='FR_autocorr_val',
    rhythmic_col='MC_rhythmic',
)

# ─── Verificación: distribución de cada categoría ─────────────────────────────
print("\n📊 FRc_wv_period_w:")
print(dbm['FRc_wv_period_w'].value_counts().to_string())

print("\n📊 FRc_n_peaks:")
print(dbm['FRc_n_peaks'].value_counts().to_string())

print("\n📊 FRc_autocorr_val:")
print(dbm['FRc_autocorr_val'].value_counts().to_string())

print("\n📊 MCc_rhythmic:")
print(dbm['MCc_rhythmic'].value_counts().to_string())

print(f"\n✅ dbm shape final: {dbm.shape}  |  index único: {dbm.index.is_unique}")

## 🗑️ Bloque 9 — Limpieza: quitar columnas no usadas en análisis final

> Antes de exportar el dbm, eliminamos las columnas FR/MC que no van a usarse en el
> análisis final. Quedan **solo 10 columnas FR/MC**:
> 6 numéricas originales (`FR_wv_period_w`, `FR_n_peaks`, `FR_mean_peak_value`,
> `FR_std_peak_value`, `FR_autocorr_val`, `MC_rhythmic`) **+ 4 categóricas**
> (`FRc_wv_period_w`, `FRc_n_peaks`, `FRc_autocorr_val`, `MCc_rhythmic`).

In [ ]:
# ─── LIMPIEZA: quitar columnas que no se usan en el análisis final ───────────
cols_to_drop = [
    "FR_hilbert_amp", "FR_hilbert_cv", "FR_autocorr_lag", "FR_rac",
    "FR_ls_pval", "FR_ls_period", "FR_rhythmic",
    "MC_period", "MC_pvalue", "MC_BH_Q",
    "FR_n_peaks_02", "FR_mean_peak_value_02", "FR_std_peak_value_02",
]

dropped   = [c for c in cols_to_drop if c in dbm.columns]
not_found = [c for c in cols_to_drop if c not in dbm.columns]
dbm = dbm.drop(columns=dropped)

print(f"🗑️  Eliminadas {len(dropped)} columnas: {dropped}")
if not_found:
    print(f"⚠️  No encontradas (ya no existían): {not_found}")
print(f"\n✅ dbm shape tras limpieza: {dbm.shape}")
print(f"\n📋 Columnas FR/MC restantes ({sum(c.startswith(('FR','MC')) for c in dbm.columns)}):")
for c in dbm.columns:
    if c.startswith(('FR','MC')):
        print(f'   - {c}')

## 💾 Exportar a Excel

In [ ]:
# ─── EXPORTAR dbm ────────────────────────────────────────────────────────────
dbm_export = dbm.reset_index()
output_save = os.path.join(BASE_DIR, 'dbm_v10.xlsx')   # incrementa la versión si quieres
dbm_export.to_excel(output_save, index=False)
print(f'💾 Guardado en: {output_save}')
print(f'   Shape: {dbm_export.shape}')

In [ ]:
# me falta quitar las columnas con LombScarg:   FR_ls_pval	FR_ls_period	FR_rhythmic +++ PREG BORJA CUALES MÁS
# del dbm y volver a sobre escribirlo para guardarlo